# 第1回：予測モデルを動かしてみる

**今日の問い：予測モデルは、データを受け取って何を返しているのか。**

上から順に実行してください。`TRY`は全員、`CHANGE`は値を1つ変える練習、
`CHALLENGE`は余裕がある人向けです。`DEEP DIVE`は経験者や自習向けの発展です。
分からないコードは、セル全体ではなく気になる数行をM365 Copilotへ貼って相談します。


In [ ]:
from pathlib import Path

def find_repo_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("pyproject.tomlがある勉強会フォルダ内で実行してください")

ROOT = find_repo_root()
DATA = ROOT / "data"
print("教材フォルダ:", ROOT)


## この回でできるようになること

- 特徴量・目的変数・学習・予測を、画面上の入出力と結びつける
- 予測を関数へ切り出し、型ヒントとassertで最小の検証を付ける
- ベースラインと比べ、設定変更の効果を交差検証の平均とばらつきで語る

### 進み方

`CORE`は同期90分で扱う本線、`DEEP DIVE`は時間があれば扱う深掘り、
`SELF-STUDY`は任意自習です。すべて終わらなくても次回へ進めます。
経験者は`CORE`を早めに終え、`DEEP DIVE`を5人で分担して読むと深まります。

### 先に押さえる言葉

- 特徴量：予測時点でモデルへ渡す情報
- 目的変数：予測したい答え
- 学習：既知データから関係を推定する処理
- 推論：学習済みモデルを未知データへ使う処理
- 並べ替え重要度：列を崩したときの性能低下で測る寄与

> **実行前の30秒予想**：今日の問いに、今の言葉で仮の答えを書いてから始めます。


In [ ]:
import pandas as pd

df = pd.read_csv(DATA / "compound_experiments.csv")
print(f"{len(df)}行 × {len(df.columns)}列")
df.head()


## まず完成済みモデルを動かす

`X`は特徴量、`y`は目的変数です。単純なベースラインと比べ、モデルが本当に価値を出しているかを最初に確かめます。


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score

features = ["molecular_weight", "logp", "tpsa", "h_bond_donors", "rotatable_bonds"]
X = df[features].fillna(df[features].median())
y = df["active"]
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
model = RandomForestClassifier(n_estimators=200, max_depth=4, random_state=42).fit(X_train, y_train)
for name, estimator in {"多数派ベースライン": baseline, "Random Forest": model}.items():
    pred = estimator.predict(X_valid)
    print(f"{name:14s} accuracy={accuracy_score(y_valid, pred):.3f}  F1={f1_score(y_valid, pred):.3f}")


## TRY：1試料の予測を見る

予測`0`は非活性、`1`は活性です。確率は確信度であり、真実そのものではありません。


In [ ]:
one_sample = X_valid.iloc[[0]]
display(one_sample)
print("予測クラス:", model.predict(one_sample)[0])
print("活性である確率:", round(model.predict_proba(one_sample)[0, 1], 3))


## CORE深掘り：評価を関数に切り出す

同じ評価を何度も書かず、型ヒントとassertで最小の検証を付けた関数にまとめます。


In [ ]:
def evaluate_classifier(estimator, X_valid, y_valid) -> dict:
    "検証データでaccuracyとF1を計算し、辞書で返す純粋な評価関数。"
    pred = estimator.predict(X_valid)
    return {
        "accuracy": round(accuracy_score(y_valid, pred), 3),
        "f1": round(f1_score(y_valid, pred), 3),
    }

scores = evaluate_classifier(model, X_valid, y_valid)
assert set(scores) == {"accuracy", "f1"}, "返す指標が想定と違います"
assert 0.0 <= scores["f1"] <= 1.0, "F1は0〜1のはず"
scores


## CHANGE

`max_depth=4`を`2`や`8`へ変え、ベースラインとの差がどう動くか記録します。

## ASK COPILOT

`fit`と`predict_proba`の違いを、測定装置の校正と未知試料の測定にたとえて説明してもらいます。

## まとめ

- 特徴量はモデルへ渡す情報、目的変数は予測したい答え
- 評価はベースラインと比べ、関数にまとめて再利用する


## DEEP DIVE：木の深さと汎化を交差検証で読む

単一の検証スコアは分割運に左右されます。交差検証で学習F1と検証F1の差（過学習）を見ます。


In [ ]:
import pandas as pd
from sklearn.model_selection import cross_validate, StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rows = []
for depth in [1, 2, 3, 4, 6, 8, None]:
    estimator = RandomForestClassifier(n_estimators=200, max_depth=depth, random_state=42)
    result = cross_validate(estimator, X, y, cv=cv, scoring="f1", return_train_score=True)
    rows.append({
        "max_depth": str(depth),
        "学習F1": result["train_score"].mean(),
        "検証F1": result["test_score"].mean(),
        "検証F1_SD": result["test_score"].std(),
    })
pd.DataFrame(rows).round(3)


### 特徴量重要度は2種類を併読する

不純度重要度は高カーディナリティ列へ偏ります。列を崩して性能低下を測る並べ替え重要度と一緒に読みます。


In [ ]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(model, X_valid, y_valid, scoring="f1", n_repeats=20, random_state=42)
importance = pd.DataFrame({
    "特徴量": features,
    "不純度重要度": model.feature_importances_,
    "並べ替え重要度": perm.importances_mean,
    "並べ替えSD": perm.importances_std,
}).sort_values("並べ替え重要度", ascending=False)
importance.round(3)


## CHALLENGE：予測確率は当たっているか（較正）

「確率0.8」の試料が本当に約80%活性かを、確率帯ごとの実際の活性率で確かめます。


In [ ]:
probability = model.predict_proba(X_valid)[:, 1]
bucket = pd.cut(probability, bins=[0, 0.2, 0.4, 0.6, 0.8, 1.0])
calibration = (
    pd.DataFrame({"確率帯": bucket, "実際の活性": y_valid.to_numpy()})
    .groupby("確率帯", observed=True)["実際の活性"]
    .agg(件数="size", 実際の活性率="mean")
)
calibration.round(3)


## よくある誤り

- 学習データの成績を実力だと思う
- 1試料の予測だけでモデル全体を判断する
- 良い数値が出るまで設定を無計画に変える

## SELF-STUDY（任意・30〜60分）

- evaluate_classifierを拡張し、precisionとrecallも返してテストを足す
- 木の深さ2・4・8を交差検証で比較し、選ぶ理由を平均とばらつきで2文書く

成果は完成したコードでなくても、予想・変更点・出力・解釈を4行で残せば十分です。

## 振り返りチェック

1. Xとyはそれぞれ何か
2. 不純度重要度と並べ替え重要度はどう違うか
3. 単一の検証スコアより交差検証を見る理由は何か

答えに詰まった項目が、次に見返す場所です。暗記ではなくNotebookの該当セルを指せればOKです。
